# MADRL Training Notebook

This notebook configures, trains, evaluates, and visualizes multi-agent energy storage experiments. Run the cells from top to bottom.


In [ ]:
from pathlib import Path
import sys

try:
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
except Exception:
    pass

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / "configs").exists():
    project_root = project_root.parent
if not (project_root / "configs").exists():
    raise RuntimeError("Could not locate the project root.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
project_root


In [ ]:
from configs import compose_experiment_config
from evaluation.plots import plot_last_k_episodes_price_action_soc
from evaluation.reward_plots import plot_reward_decomposition
from madrl.notebook_utils import (
    build_runner,
    get_madrl_checkpoint_root,
    evaluate_runner,
    get_lstm_artifact_paths,
    inspect_runner_io,
    summarize_cfg,
)


In [ ]:
profile = "fast_train"          # "base" | "debug" | "fast_train"
algorithm = "MADDPG"            # "MADDPG" | "MATD3"
model_family = "mlp"            # "mlp" | "transformer" | "graph"
reward_type = "composite"       # "composite" | "sparse"
observation_profile = "default" # "default" | "minimal" | "local_only"
local_features = None            # 可选覆写，例如 ["time", "price", "load", "soc"]
sequence_features = None         # 可选覆写，例如 ["price", "load"]
forecast_type = "perfect"       # "perfect" | "naive" | "lstm"
vec_env_type = None                 # None | "dummy" | "subproc"
seed = 0
n_eval_episodes = 2
reward_plot_window = 20         # 奖励曲线移动平均窗口
n_recent_episodes_to_plot = 2   # 展示最近若干条 episode 轨迹


In [ ]:
cfg = compose_experiment_config(
    profile=profile,
    algorithm=algorithm,
    model_family=model_family,
    reward_type=reward_type,
    observation_profile=observation_profile,
    forecast_type=forecast_type,
    vec_env_type=vec_env_type,
    local_features=local_features,
    sequence_features=sequence_features,
)

if forecast_type == "lstm":
    cfg.forecast.lstm_model_path = get_lstm_artifact_paths(project_root)["model_path"]

summary = summarize_cfg(cfg)
summary


In [ ]:
runner = build_runner(cfg, seed=seed, env_name="NotebookTrain", number=1)
plot_reward_fn = runner.env_evaluate.reward_fn
print("观测 schema =", cfg.runtime.observation_schema)
print("观测 layout =", cfg.runtime.observation_layout)
print("动作维度 =", cfg.runtime.action_dim)


In [ ]:
sanity_summary = inspect_runner_io(runner, cfg)
sanity_summary


In [ ]:
episodes_completed = runner.run()
print("训练完成 episode 数 =", episodes_completed)
runner.perf_summary


In [ ]:
eval_results = evaluate_runner(runner, cfg, n_episodes=n_eval_episodes, deterministic=True)
print("平均评估回报 =", eval_results["mean_episode_reward"])

eval_histories = eval_results.get("histories", [])
eval_window = min(reward_plot_window, max(1, len(eval_results["episode_rewards"])))

plot_reward_decomposition(
    history=runner.history,
    episode_rewards=runner.episode_rewards,
    reward_fn=plot_reward_fn,
    title="训练阶段奖励分解",
    window=reward_plot_window,
)

plot_last_k_episodes_price_action_soc(
    history=runner.history,
    k=n_recent_episodes_to_plot,
    n_agents=cfg.env.num_agents,
    title_prefix="训练",
)

plot_reward_decomposition(
    history=eval_histories,
    episode_rewards=eval_results["episode_rewards"],
    reward_fn=plot_reward_fn,
    title="评估阶段奖励分解",
    window=eval_window,
)

plot_last_k_episodes_price_action_soc(
    history=eval_histories,
    k=min(n_recent_episodes_to_plot, len(eval_histories)),
    n_agents=cfg.env.num_agents,
    title_prefix="评估",
)


In [ ]:
save_dir = get_madrl_checkpoint_root(project_root)
save_dir.mkdir(parents=True, exist_ok=True)
runner.save_model(str(save_dir), episode=episodes_completed)

final_summary = {
    "save_dir": str(save_dir),
    "saved_episode": int(episodes_completed),
    "mean_eval_reward": float(eval_results["mean_episode_reward"]),
    "perf_summary": runner.perf_summary,
}
print("模型已保存到 =", save_dir)
runner.close()
final_summary
